### Name: Muhammad Armughan Bakhshi
### Class: Ds-4B
### ID : 24i-2626

Github Link: https://github.com/Armubakh/UNO-Assignment.git

In [19]:
import random
import copy
import tkinter as tk

COLORS = ['Red', 'Blue', 'Green', 'Yellow'] 
VALUES = [str(i) for i in range(10)] + ['Skip'] 

class Card: 
    def __init__(self, color, value):
        self.color = color
        self.value = value

    def __repr__(self):
        return f"{self.color} {self.value}"

#  Deck Generation 
def generate_deck():
    deck = [Card(color, value) for color in COLORS for value in VALUES]
    random.shuffle(deck)
    return deck

def get_valid_moves(hand, top_card):
    valid_moves = []
    for card in hand:
        
        # Rule 1: A player may play a card if it matches the color OR number 
        if card.color == top_card.color or card.value == top_card.value:
            valid_moves.append(card)
            
    return valid_moves

# State Transition 
def apply_move(state, player_key, move):
    new_state = copy.deepcopy(state)
    
    # Rule 2: If no valid card exists, player must draw 1 card 
    if move == "Draw":
        if len(new_state['deck']) > 0:
            drawn_card = new_state['deck'].pop()
            new_state[player_key].append(drawn_card)
    else:
        for i, card in enumerate(new_state[player_key]):
            if card.color == move.color and card.value == move.value:
                new_state[player_key].pop(i)
                break
        new_state['top_card'] = move
        
    return new_state

def simulate_random_game(initial_state):
    c_st = copy.deepcopy(initial_state)
    p = ['Player1', 'Player2', 'Ai']
    t_idx = 0

    print(f"\ntop card: {c_st['top_card']}\n")
    print("--- game tree ---")
    p_tr(c_st, t_idx)

    while True:
        c_p = p[t_idx % 3]
        hnd = c_st[c_p]

        # Rule 4
        if len(hnd) == 0:
            print(f"\ngame over! {c_p} wins!")
            break

        print(f"{c_p} hand:")
        for i, c in enumerate(hnd):
            print(f"[{i}] {c}")
            
        print(f"\n{c_p} decision(all possible decisions considered at depth 1):\n")
        
        if c_p == 'Player1':
            _, c_m = mm(c_st, 3, 1, c_p, t_idx)
        elif c_p == 'Player2':
            _, c_m = em(c_st, 3, c_p, t_idx)
        else:
            _, c_m = mm(c_st, 3, 1, c_p, t_idx)
            
        if not c_m:
            c_m = "Draw"
            
        c_st = apply_move(c_st, c_p, c_m)
        
        # Rule 3: if player plays skip, next player's turn is skipped 
        if c_m != "Draw" and c_m.value == 'Skip':
            t_idx += 2 
        else:
            t_idx += 1 
            
        if len(c_st['deck']) == 0:
            print("\ndraw deck is empty. it's a tie!")
            break

# Implementation of evaluation Function
def evaluate_state(st, p, strat="baseline"):
    ops = [x for x in ['Player1', 'Player2', 'Ai'] if x != p]
    
    my_c = len(st[p])
    op_c = (len(st[ops[0]]) + len(st[ops[1]])) / 2
    
    skp = sum(1 for c in st[p] if c.value == 'Skip')
    
    w_my, w_op, w_sk = 5, 2, 3
    
    if strat == "offensive":
        w_my, w_op, w_sk = 7, 2, 2
    elif strat == "defensive":
        w_my, w_op, w_sk = 4, 4, 5
        
    return 50 - (w_my * my_c) + (w_op * op_c) + (w_sk * skp)

# Minimax and Expectimax implementation
def mm(st, dp, is_mx, p_k, t_idx):
    if dp == 0 or not st['Player1'] or not st['Player2'] or not st['Ai']:
        return evaluate_state(st, p_k, "defensive"), 0
        
    p = ['Player1', 'Player2', 'Ai']
    c_p = p[t_idx % 3]
    v_m = get_valid_moves(st[c_p], st['top_card'])
    
    if not v_m:
        v_m = ["Draw"]
        
    b_m = 0
    
    if is_mx:
        
        m_v = -float('inf')
        
        for m in v_m:
            n_st = apply_move(st, c_p, m)
            n_t = t_idx + 2 if m != "Draw" and m.value == 'Skip' else t_idx + 1
            v, _ = mm(n_st, dp - 1, 0, p_k, n_t)
            
            if dp == 3:
                print(f"play: {m}")
                print(f"expected score: {round(v, 1)}\n")
                
            if v > m_v:
                m_v = v
                b_m = m
        return m_v, b_m
    else:
        m_v = float('inf')
        for m in v_m:
            n_st = apply_move(st, c_p, m)
            n_t = t_idx + 2 if m != "Draw" and m.value == 'Skip' else t_idx + 1
            n_p = p[n_t % 3]
            n_mx = 1 if n_p == p_k else 0
            v, _ = mm(n_st, dp - 1, n_mx, p_k, n_t)
            if v < m_v:
                m_v = v
                b_m = m
        return m_v, b_m
        
def em(st, dp, p_k, t_idx):
    if dp == 0 or not st['Player1'] or not st['Player2'] or not st['Ai']:
        return evaluate_state(st, p_k, "offensive"), 0
        
    p = ['Player1', 'Player2', 'Ai']
    c_p = p[t_idx % 3]
    v_m = get_valid_moves(st[c_p], st['top_card'])
    
    if not v_m:
        v_m = ["Draw"]
        
    b_m = 0
    
    if c_p == p_k:
        m_v = -float('inf')
        
        for m in v_m:
            if m == "Draw":
                e_v = 0
                d_c = len(st['deck'])
                if d_c > 0:
                    pr = 1.0 / d_c
                    for i in range(d_c):
                        n_st = copy.deepcopy(st)
                        d_o = n_st['deck'].pop(i)
                        n_st[c_p].append(d_o)
                        v, _ = em(n_st, dp - 1, p_k, t_idx + 1)
                        e_v += pr * v
                        
                    if dp == 3:
                        print(f"play: draw")
                        print(f"expected score: {round(e_v, 1)}\n")
                        
                    if e_v > m_v:
                        m_v = e_v
                        b_m = m
                else:
                    v = evaluate_state(st, p_k, "offensive")
                    
                    if dp == 3:
                        print(f"play: draw")
                        print(f"expected score: {round(v, 1)}\n")
                        
                    if v > m_v:
                        m_v = v
                        b_m = m
            else:
                n_st = apply_move(st, c_p, m)
                n_t = t_idx + 2 if m.value == 'Skip' else t_idx + 1
                v, _ = em(n_st, dp - 1, p_k, n_t)
                
                if dp == 3:
                    print(f"play: {m}")
                    print(f"expected score: {round(v, 1)}\n")
                    
                if v > m_v:
                    m_v = v
                    b_m = m
        return m_v, b_m
    else:
        e_v = 0
        l_m = len(v_m)
        for m in v_m:
            n_st = apply_move(st, c_p, m)
            n_t = t_idx + 2 if m != "Draw" and m.value == 'Skip' else t_idx + 1
            v, _ = em(n_st, dp - 1, p_k, n_t)
            e_v += v
        return (e_v / l_m) if l_m > 0 else 0, 0 
        
# GUI 
def draw_card_gui(canvas, card_obj, x, y, hidden=False):
    if hidden:
        canvas.create_rectangle(x-2, y-2, x+62, y+92, fill='white', outline='black', width=2)
        canvas.create_rectangle(x, y, x+60, y+90, fill='#141414', outline='')
        return
        
    color_map = {'Red': '#ff5555', 'Blue': '#5555ff', 'Green': '#55ff55', 'Yellow': '#ffff55'}
    color = color_map.get(card_obj.color, '#aaaaaa')
    val = card_obj.value
    
    canvas.create_rectangle(x-2, y-2, x+62, y+92, fill='white', outline='black', width=2)
    canvas.create_rectangle(x, y, x+60, y+90, fill=color, outline='')
    canvas.create_text(x+30, y+45, text=val, fill='black', font=('Arial', 16, 'bold'))

def run_game(mode, root):
    for widget in root.winfo_children():
        widget.destroy()
        
    root.geometry("1000x700")
    root.title(f"UNO AI - {mode.capitalize()} Mode (Depth 3)")
    
    canvas = tk.Canvas(root, width=1000, height=700, bg='#1e1e2e')
    canvas.pack(fill="both", expand=True)
    
    deck = generate_deck()
    state = {
        'Player1': [deck.pop() for _ in range(5)],
        'Player2': [deck.pop() for _ in range(5)],
        'Ai': [deck.pop() for _ in range(5)],
        'deck': deck,
        'top_card': deck.pop()
    }
    turn_idx = [2]
    
    display_names = {'Player1': 'Defensive', 'Player2': 'Offensive', 'Ai': 'AI'}
    
    def update_board():
        canvas.delete("all")
        if not state['Player1'] or not state['Player2'] or not state['Ai']:
            winner_key = 'Player1' if not state['Player1'] else ('Player2' if not state['Player2'] else 'Ai')
            canvas.create_text(500, 350, text=f"Game Over! {display_names[winner_key]} Wins!", fill='white', font=('Arial', 32, 'bold'))
            return

        draw_card_gui(canvas, state['top_card'], 500 - 30, 350 - 45)
        
        deck_count = len(state['deck'])
        canvas.create_rectangle(380, 303, 444, 397, fill='white', outline='black', width=2)
        canvas.create_rectangle(382, 305, 442, 395, fill='#141414', outline='')
        canvas.create_text(412, 350, text=f"{deck_count}", fill='white', font=('Arial', 20, 'bold'))
        
        for i, card in enumerate(state['Player1']):
            draw_card_gui(canvas, card, 50 + i * 50, 50, hidden=(mode == "manual"))
            
        for i, card in enumerate(state['Player2']):
            draw_card_gui(canvas, card, 700 + i * 50, 50, hidden=(mode == "manual"))
            
        start_x = (1000 - (len(state['Ai']) * 70)) // 2
        for i, card in enumerate(state['Ai']):
            draw_card_gui(canvas, card, start_x + i * 70, 700 - 110)

        players = ['Player1', 'Player2', 'Ai']
        curr_player = players[turn_idx[0] % 3]
        canvas.create_text(500, 20, text=f"Turn: {display_names[curr_player]}", fill='white', font=('Arial', 16))

        if curr_player == 'Ai' and mode == "manual":
            valid_moves = get_valid_moves(state['Ai'], state['top_card'])
            if not valid_moves:
                canvas.create_rectangle(450, 500, 550, 540, fill='gray')
                canvas.create_text(500, 520, text="Draw Card", fill='black')

    def handle_click(e):
        if mode == "simulation" or (turn_idx[0] % 3) != 2:
            return
            
        valid_moves = get_valid_moves(state['Ai'], state['top_card'])
        
        if not valid_moves and 450 <= e.x <= 550 and 500 <= e.y <= 540:
            state.update(apply_move(state, 'Ai', "Draw"))
            turn_idx[0] += 1
            update_board()
            canvas.after(500, ai_turn)
            return

        start_x = (1000 - (len(state['Ai']) * 70)) // 2
        for i, card in enumerate(state['Ai']):
            x_pos = start_x + i * 70
            if x_pos <= e.x <= x_pos + 60 and 590 <= e.y <= 680:
                if card in valid_moves:
                    state.update(apply_move(state, 'Ai', card))
                    turn_idx[0] += 2 if card.value == 'Skip' else 1
                    update_board()
                    canvas.after(500, ai_turn)
                break

    def ai_turn():
        players = ['Player1', 'Player2', 'Ai']
        curr_player = players[turn_idx[0] % 3]
        
        if not state['Player1'] or not state['Player2'] or not state['Ai']:
            return

        if curr_player == 'Ai' and mode == "manual":
            return

        canvas.create_text(500, 280, text=f"{display_names[curr_player]} is thinking...", fill='yellow', font=('Arial', 14))
        root.update()
        
        if curr_player == 'Player1':
            _, best_move = mm(state, 3, 1, curr_player, turn_idx[0])
        elif curr_player == 'Player2':
            _, best_move = em(state, 3, curr_player, turn_idx[0])
        else:
            _, best_move = mm(state, 3, 1, curr_player, turn_idx[0])
            
        if not best_move:
            best_move = "Draw"
            
        state.update(apply_move(state, curr_player, best_move))
        turn_idx[0] += 2 if best_move != "Draw" and best_move.value == 'Skip' else 1
        
        update_board()
        canvas.after(1000, ai_turn)

    canvas.bind("<Button-1>", handle_click)
    update_board()
    
    if mode == "simulation" or (turn_idx[0] % 3) != 2:
        canvas.after(1000, ai_turn)

def main_menu():
    root = tk.Tk()
    root.title("UNO AI Setup")
    root.geometry("300x200")
    
    tk.Label(root, text="Select Mode:").pack(pady=10)
    game_mode = tk.StringVar(value="manual")
    tk.Radiobutton(root, text="Manual (Play vs AI)", variable=game_mode, value="manual").pack()
    tk.Radiobutton(root, text="Simulation (AI vs AI)", variable=game_mode, value="simulation").pack()
    
    tk.Label(root, text="Search depth is fixed to 3").pack(pady=10)
    
    def start_btn():
        run_game(game_mode.get(), root)
        
    tk.Button(root, text="Start Game", command=start_btn).pack(pady=10)
    root.mainloop()

if __name__ == "__main__":
    main_menu()
def p_tr(st, t_idx):
    p = ['Player1', 'Player2', 'Ai']
    p0 = p[t_idx % 3]

    v1 = get_valid_moves(st[p0], st['top_card'])
    m1 = v1[0] if v1 else "Draw"

    st2 = copy.deepcopy(st)
    if m1 != "Draw":
        st2 = apply_move(st2, p0, m1)
    t2 = (t_idx + 2) if (m1 != "Draw" and m1.value == 'Skip') else (t_idx + 1)
    p1 = p[t2 % 3]

    v2 = get_valid_moves(st2[p1], st2['top_card'])
    m2 = v2[0] if v2 else "Draw"

    st3 = copy.deepcopy(st2)
    if m2 != "Draw":
        st3 = apply_move(st3, p1, m2)
    t3 = (t2 + 2) if (m2 != "Draw" and m2.value == 'Skip') else (t2 + 1)
    p2 = p[t3 % 3]

    c1 = f"play {m1.color[0]}{m1.value}" if m1 != "Draw" else "Draw"
    c2 = f"play {m2.color[0]}{m2.value}" if m2 != "Draw" else "Draw"

    print(f"\n{p0.center(42)}")
    print("                /            \\")
    print(f"{c1.center(22)}{'Draw'.center(20)}")
    print("              |                |")

    n1 = p1 if m1 != "Draw" else "Chance"
    print(f"{n1.center(24)}{'Chance'.center(16)}")

    if m1 != "Draw":
        print("           /      \\            |")
        print(f"{c2.center(18)}{'Draw'.center(12)}{'cards...'.center(14)}")
        print("           |        |")
        n2 = p2 if m2 != "Draw" else "Chance"
        print(f"{n2.center(20)}{'Chance'.center(10)}")
        print("          / \\       |")
        print("        pld drw    ...")
    else:
        print("              |                |")
        print("          cards...         cards...")
    print("\n")

# GUI
 # testing if game works           
game_deck = generate_deck()

test_state = {
    'Player1': [game_deck.pop() for _ in range(5)],
    'Player2': [game_deck.pop() for _ in range(5)],
    'Ai': [game_deck.pop() for _ in range(5)],
    'top_card': game_deck.pop(),
    'deck': game_deck
}

simulate_random_game(test_state)

play: Green 8
expected score: 58.0

play: Blue 8
expected score: 50.0

play: Draw
expected score: 44.0

play: draw
expected score: 17.0

play: Green 5
expected score: 62.0

play: Green 6
expected score: 58.0

play: Blue 8
expected score: 58.0

play: Draw
expected score: 41.0

play: Blue 5
expected score: 23.0

play: Blue 0
expected score: 62.0

play: Blue 8
expected score: 62.0

play: Yellow 0
expected score: 45.0

play: Blue 6
expected score: 41.0

play: Blue Skip
expected score: 42.0

play: Yellow 7
expected score: 30.0

play: Yellow 2
expected score: 30.0

play: Draw
expected score: 54.0

play: Yellow 9
expected score: 45.0

play: Yellow 1
expected score: 49.0

play: Yellow 3
expected score: 49.0

play: Yellow 2
expected score: 37.0

play: Draw
expected score: 51.0

play: Yellow 9
expected score: 49.0

play: Yellow 3
expected score: 53.0

play: draw
expected score: 25.8

play: Yellow Skip
expected score: 52.0

play: draw
expected score: 22.2

play: Draw
expected score: 48.0

play: Y

the function calculates a score using this baseline formula :

### score = 50 - w_my(c_ai) + w_op(c_opp) + w_sk(s)

#### the variables:

c_ai: cards in the ai's hand. this is subtracted because having fewer cards brings you closer to winning.

c_opp: average cards held by the two opponents. this is added because it is good when opponents have a lot of cards.

s: skip cards held. this is added because skip cards give you turn control.

50: a constant value to keep the overall score positive.


#### the strategy weights:

baseline (w_my=5, w_op=2, w_sk=3): standard, balanced play.

offensive (w_my=7, w_op=2, w_sk=2): heavily penalizes holding your own cards to force aggressive card shedding.

defensive (w_my=4, w_op=4, w_sk=5): strongly rewards hoarding skip cards and keeping opponent card counts high.

### Conclusion: Algorithm Comparison

#### The Strategy:

Minimax (Player 1):
Defensive. Assumes opponents always play perfectly to maximize Player 1's penalties. Prioritizes hoarding 'Skip' cards and maintaining turn control over emptying its hand.

Expectimax (Player 2): 
Offensive. Treats the 'Draw' action as a Chance node, calculating the mathematical probability of deck outcomes. Prioritizes aggressive card shedding.

### Which Algorithm Performed Best?

Expectimax consistently achieved a higher win rate.

### The Reasoning:

UNO is a game of imperfect information and chance, unlike perfect-information games like Chess. Minimax fails here because it avoids mathematically sound plays just to prevent worst-case scenarios (like forced draws). Expectimax succeeds because it takes calculated risks based on actual deck probabilities, resulting in a much more efficient and aggressive playstyle.